# LeetCode 1765 — Map of Highest Peak bằng Python

Link đề: https://leetcode.com/problems/map-of-highest-peak/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def highestPeak(self, isWater: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng chiều cao tối đa của mỗi ô bị chặn bởi **khoảng cách tới ô nước gần nhất**, và gán đúng khoảng cách đó là tối ưu.
2. Cài **BFS đa nguồn** (multi-source BFS): đẩy **mọi** ô nước vào hàng đợi ngay từ đầu.
3. Cài cách **quy hoạch động hai lượt** (trên-trái rồi dưới-phải) cho khoảng cách Manhattan.
4. Viết **hàm kiểm tra hợp lệ** cho bài có nhiều đáp án (mọi phép gán hợp lệ có cùng giá trị max tối ưu).
5. Biết đây là cùng bài với **542. 01 Matrix**.

## 2. Đề bài & ví dụ

Cho ma trận `isWater` kích thước `m × n`: `isWater[i][j] == 0` là **đất**, `== 1` là **nước**. Gán cho mỗi ô một **chiều cao** thỏa:
- chiều cao **không âm**;
- ô nước có chiều cao **0**;
- hai ô **kề cạnh** (bắc, nam, đông, tây) chênh nhau **tối đa 1**.

Tìm phép gán sao cho **chiều cao lớn nhất** trong ma trận là **lớn nhất có thể**. Trả về ma trận chiều cao; nếu có nhiều đáp án, trả về bất kỳ đáp án nào.

**Ví dụ 1:** `isWater = [[0,1],[0,0]]` → `[[1,0],[2,1]]`.

**Ví dụ 2:** `isWater = [[0,0,1],[1,0,0],[0,0,0]]` → `[[1,1,0],[0,1,1],[1,2,2]]` (max = 2; mọi phép gán hợp lệ có max = 2 đều được chấp nhận).

**Ràng buộc:** `1 ≤ m, n ≤ 1000`; có **ít nhất một** ô nước.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [([[0, 1], [0, 0]], [[1, 0], [2, 1]]),
            ([[0, 0, 1], [1, 0, 0], [0, 0, 0]], [[1, 1, 0], [0, 1, 1], [1, 2, 2]])]
for g, exp in examples:
    print("isWater =", g, "-> một đáp án:", exp)

isWater = [[0, 1], [0, 0]] -> một đáp án: [[1, 0], [2, 1]]
isWater = [[0, 0, 1], [1, 0, 0], [0, 0, 0]] -> một đáp án: [[1, 1, 0], [0, 1, 1], [1, 2, 2]]


## 3. Kiến thức nền

### 3.1 Duyệt 4 hướng trong ma trận

Dùng danh sách hướng `DIRS = [(1,0),(-1,0),(0,1),(0,-1)]` và kiểm tra biên `0 ≤ x < m`, `0 ≤ y < n`.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
def neighbors(i, j, m, n):
    for dx, dy in DIRS:
        x, y = i + dx, j + dy
        if 0 <= x < m and 0 <= y < n:
            yield x, y
print(list(neighbors(0, 0, 3, 3)), list(neighbors(1, 1, 3, 3)))

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2 `collections.deque` làm hàng đợi

`append` thêm cuối, `popleft` lấy đầu — đều `O(1)`. (`list.pop(0)` là `O(n)`, không dùng cho BFS.)

In [3]:
q = deque([(0, 0)]); q.append((0, 1)); print(q.popleft(), q)

(0, 0) deque([(0, 1)])


### 3.3 BFS cho khoảng cách ngắn nhất trên lưới không trọng số

BFS thăm các ô theo **thứ tự khoảng cách tăng dần** từ nguồn: lần đầu tiên chạm tới một ô là qua đường ngắn nhất. Đánh dấu "đã thăm" ngay khi **đẩy vào** hàng đợi để không đẩy trùng.

### 3.4 Khoảng cách Manhattan

Trên lưới không có vật cản, đường ngắn nhất 4 hướng giữa `(i, j)` và `(a, b)` dài đúng `|i − a| + |j − b|`.

In [4]:
print(abs(2 - 0) + abs(1 - 2), "= khoảng cách từ (2,1) tới (0,2)")

3 = khoảng cách từ (2,1) tới (0,2)


### 3.5 Hàm kiểm tra hợp lệ cho bài nhiều đáp án

Vì đề chấp nhận **bất kỳ** đáp án tối ưu nào, ta không so sánh ma trận trực tiếp mà kiểm tra: (1) nước = 0, (2) không âm, (3) kề nhau chênh ≤ 1, (4) max đạt giá trị tối ưu.

In [5]:
def is_valid(isWater, H, best_max):
    m, n = len(isWater), len(isWater[0])
    if len(H) != m or any(len(r) != n for r in H):
        return False
    for i in range(m):
        for j in range(n):
            if H[i][j] < 0 or (isWater[i][j] == 1 and H[i][j] != 0):
                return False
            for x, y in neighbors(i, j, m, n):
                if abs(H[i][j] - H[x][y]) > 1:
                    return False
    return max(map(max, H)) == best_max

print(is_valid(*examples[1], 2), is_valid(examples[1][0], [[0, 0, 0], [0, 0, 0], [0, 0, 0]], 2))

True False


## 4. Ý tưởng nền tảng / trực giác

**Cận trên:** đi từ ô `c` tới ô nước gần nhất `w` qua `d` bước; mỗi bước chiều cao giảm tối đa 1 và ô nước cao 0 ⇒ `H[c] ≤ d = dist(c, nước gần nhất)`.

**Đạt cận:** gán `H[c] = dist(c)`. Ô nước có `dist = 0`; hai ô kề nhau có `dist` chênh tối đa 1 (vì đi thêm một bước) ⇒ hợp lệ. Mỗi ô đều đạt cận trên riêng của nó ⇒ max cũng tối ưu.

Tính tay **Ví dụ 2** (nước ở `(0,2)` và `(1,0)`):

| ô | khoảng cách tới (0,2) | tới (1,0) | min |
|---|---|---|---|
| (0,0) | 2 | 1 | 1 |
| (0,1) | 1 | 2 | 1 |
| (1,1) | 2 | 1 | 1 |
| (1,2) | 1 | 2 | 1 |
| (2,0) | 4 | 1 | 1 |
| (2,1) | 3 | 2 | 2 |
| (2,2) | 2 | 3 | 2 |

## 5. Cách 1 — Brute Force / baseline: mỗi ô tính min khoảng cách Manhattan tới mọi ô nước

In [6]:
def peak_bruteforce(isWater):
    m, n = len(isWater), len(isWater[0])
    water = [(i, j) for i in range(m) for j in range(n) if isWater[i][j]]
    return [[min(abs(i - a) + abs(j - b) for a, b in water) for j in range(n)] for i in range(m)]

for g, exp in examples:
    got = peak_bruteforce(g); print(got, "== mẫu?", got == exp); assert got == exp

[[1, 0], [2, 1]] == mẫu? True
[[1, 1, 0], [0, 1, 1], [1, 2, 2]] == mẫu? True


**Độ phức tạp:** `O(m·n·W)` với `W` = số ô nước — tệ nhất `10⁶ · 5·10⁵`: **không thể**.

**Nút thắt:** mỗi ô lại quét toàn bộ ô nước, trong khi khoảng cách của ô kề nhau gần như giống nhau — có thể **lan truyền** từ ô này sang ô kia.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (BFS đa nguồn):** thay vì BFS từ từng ô nước (tốn `W` lần), hãy coi **tất cả** ô nước là một "siêu nguồn": đẩy hết vào hàng đợi với khoảng cách 0, rồi BFS một lần. Ô nào được chạm đầu tiên thì khoảng cách của nó là khoảng cách tới ô nước **gần nhất**.

**Quy tắc lan truyền:** lấy ô `(i, j)` khỏi hàng đợi; với mỗi ô kề `(x, y)` **chưa có giá trị**, đặt `H[x][y] = H[i][j] + 1` và đẩy vào hàng đợi.

**Invariant BFS:** các ô trong hàng đợi có khoảng cách `d` hoặc `d + 1` theo thứ tự không giảm ⇒ khi gán `H[x][y]` lần đầu, giá trị đó là nhỏ nhất.

**Insight 2 (DP hai lượt):** trên lưới không vật cản, khoảng cách là Manhattan. Một lượt quét từ trên-trái xuống dưới-phải lan truyền mọi bước đi **xuống** hoặc **sang phải**; một lượt ngược lại lan truyền mọi bước **lên** hoặc **sang trái**. Mọi đường ngắn nhất có thể sắp lại thành "một đoạn do lượt 1 lo, tiếp theo một đoạn do lượt 2 lo" (xem mục 7.2) ⇒ hai lượt là đủ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS đa nguồn

In [7]:
def peak_bfs(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]
    q = deque()
    for i in range(m):
        for j in range(n):
            if isWater[i][j]:
                H[i][j] = 0
                q.append((i, j))
    while q:
        i, j = q.popleft()
        for dx, dy in DIRS:
            x, y = i + dx, j + dy
            if 0 <= x < m and 0 <= y < n and H[x][y] == -1:
                H[x][y] = H[i][j] + 1
                q.append((x, y))
    return H

for g, exp in examples:
    print(peak_bfs(g))

[[1, 0], [2, 1]]
[[1, 1, 0], [0, 1, 1], [1, 2, 2]]


**Code nộp LeetCode (Cách 2):** xem mục 9. **Độ phức tạp:** mỗi ô vào hàng đợi đúng một lần, xét 4 ô kề ⇒ Time `O(m·n)`, Space `O(m·n)`.

### 7.2 Cách 3 — Quy hoạch động hai lượt

**Giải thích từng đoạn:**
1. Khởi tạo `H = 0` cho nước, `∞` cho đất.
2. Lượt 1 (từ trên-trái xuống dưới-phải): `H[i][j] = min(H[i][j], H[i−1][j] + 1, H[i][j−1] + 1)`.
3. Lượt 2 (từ dưới-phải lên trên-trái): `H[i][j] = min(H[i][j], H[i+1][j] + 1, H[i][j+1] + 1)`.

In [8]:
def peak_dp(isWater):
    m, n = len(isWater), len(isWater[0])
    INF = m + n
    H = [[0 if isWater[i][j] else INF for j in range(n)] for i in range(m)]
    for i in range(m):
        for j in range(n):
            if i > 0: H[i][j] = min(H[i][j], H[i - 1][j] + 1)
            if j > 0: H[i][j] = min(H[i][j], H[i][j - 1] + 1)
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if i < m - 1: H[i][j] = min(H[i][j], H[i + 1][j] + 1)
            if j < n - 1: H[i][j] = min(H[i][j], H[i][j + 1] + 1)
    return H

print([peak_dp(g) for g, _ in examples])

[[[1, 0], [2, 1]], [[1, 1, 0], [0, 1, 1], [1, 2, 2]]]


**Vì sao hai lượt đủ:** xét ô `c = (ci, cj)` và ô nước gần nhất `w = (wi, wj)`. Lượt 1 lan truyền theo bước "xuống" (`i` tăng) và "sang phải" (`j` tăng); lượt 2 theo bước "lên" và "sang trái". Chọn đường ngắn nhất theo bốn trường hợp:
- `w` ở trên-trái (`wi ≤ ci`, `wj ≤ cj`): toàn bộ đường đi xuống/phải ⇒ lượt 1 xử lý hết.
- `w` ở dưới-phải: toàn bộ lên/trái ⇒ lượt 2.
- `w` ở trên-phải (`wi ≤ ci`, `wj ≥ cj`): đi **xuống** theo cột `wj` tới hàng `ci` (lượt 1), rồi **sang trái** theo hàng `ci` tới `c` (lượt 2, chạy sau lượt 1).
- `w` ở dưới-trái: đi **sang phải** theo hàng `wi` tới cột `cj` (lượt 1), rồi **lên** theo cột `cj` tới `c` (lượt 2).

Trong mọi trường hợp, giá trị đúng được lan tới `c` ⇒ `H[c] ≤ d(c)`; và DP không bao giờ tạo giá trị nhỏ hơn khoảng cách thật ⇒ `H[c] = d(c)`. Bộ test ở mục 12 còn đối chiếu với vét cạn.

**Code nộp LeetCode (Cách 3):** đổi tên thành `highestPeak`. **Độ phức tạp:** Time `O(m·n)`, Space `O(1)` ngoài kết quả.

### 7.3 Cách 4 — BFS theo **tầng** bằng list (không dùng deque)

Thay hàng đợi bằng list `frontier` của tầng hiện tại; xây `nxt` cho tầng tiếp theo. Cách này giúp thấy rõ "tầng = chiều cao".

In [9]:
class SolutionLayers:
    def highestPeak(self, isWater: List[List[int]]) -> List[List[int]]:
        m, n = len(isWater), len(isWater[0])
        H = [[-1] * n for _ in range(m)]
        frontier = [(i, j) for i in range(m) for j in range(n) if isWater[i][j]]
        for i, j in frontier:
            H[i][j] = 0
        h = 0
        while frontier:
            h += 1
            nxt = []
            for i, j in frontier:
                for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    x, y = i + dx, j + dy
                    if 0 <= x < m and 0 <= y < n and H[x][y] == -1:
                        H[x][y] = h
                        nxt.append((x, y))
            frontier = nxt
        return H

print([SolutionLayers().highestPeak(g) for g, _ in examples])

[[[1, 0], [2, 1]], [[1, 1, 0], [0, 1, 1], [1, 2, 2]]]


**Độ phức tạp Cách 4:** như Cách 2.

## 8. TRACE từng bước

Trace BFS theo tầng trên **Ví dụ 2** (in ma trận sau mỗi tầng, `.` là chưa gán):

In [10]:
def trace_layers(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]
    frontier = [(i, j) for i in range(m) for j in range(n) if isWater[i][j]]
    for i, j in frontier: H[i][j] = 0
    h = 0
    def show():
        for row in H: print("   ", " ".join("." if v < 0 else str(v) for v in row))
    print("tầng 0 (nước):", frontier); show()
    while frontier:
        h += 1; nxt = []
        for i, j in frontier:
            for x, y in neighbors(i, j, m, n):
                if H[x][y] == -1:
                    H[x][y] = h; nxt.append((x, y))
        frontier = nxt
        if frontier:
            print(f"tầng {h}:", frontier); show()
    return H

H = trace_layers(examples[1][0]); assert H == examples[1][1]

tầng 0 (nước): [(0, 2), (1, 0)]
    . . 0
    0 . .
    . . .
tầng 1: [(1, 2), (0, 1), (2, 0), (0, 0), (1, 1)]
    1 1 0
    0 1 1
    1 . .
tầng 2: [(2, 2), (2, 1)]
    1 1 0
    0 1 1
    1 2 2


| tầng | ô được gán | ý nghĩa |
|---|---|---|
| 0 | (0,2), (1,0) | ô nước |
| 1 | (1,2), (0,1), (2,0), (0,0), (1,1) | cách nước 1 bước |
| 2 | (2,2), (2,1) | cách nước 2 bước |

Trace Cách 3 (hai lượt) trên Ví dụ 1:

In [11]:
g = examples[0][0]; m, n = 2, 2; INF = 9
H = [[0 if g[i][j] else INF for j in range(n)] for i in range(m)]
print("khởi tạo:", H)
for i in range(m):
    for j in range(n):
        if i > 0: H[i][j] = min(H[i][j], H[i - 1][j] + 1)
        if j > 0: H[i][j] = min(H[i][j], H[i][j - 1] + 1)
print("sau lượt 1:", H)
for i in range(m - 1, -1, -1):
    for j in range(n - 1, -1, -1):
        if i < m - 1: H[i][j] = min(H[i][j], H[i + 1][j] + 1)
        if j < n - 1: H[i][j] = min(H[i][j], H[i][j + 1] + 1)
print("sau lượt 2:", H); assert H == examples[0][1]

khởi tạo: [[9, 0], [9, 9]]
sau lượt 1: [[9, 0], [9, 1]]
sau lượt 2: [[1, 0], [2, 1]]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — BFS đa nguồn.

In [12]:
class Solution:
    def highestPeak(self, isWater: List[List[int]]) -> List[List[int]]:
        m, n = len(isWater), len(isWater[0])
        H = [[-1] * n for _ in range(m)]
        q = deque()
        for i in range(m):
            for j in range(n):
                if isWater[i][j] == 1:
                    H[i][j] = 0
                    q.append((i, j))
        while q:
            i, j = q.popleft()
            for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                x, y = i + dx, j + dy
                if 0 <= x < m and 0 <= y < n and H[x][y] == -1:
                    H[x][y] = H[i][j] + 1
                    q.append((x, y))
        return H

for g, exp in examples:
    got = Solution().highestPeak(g); print(got); assert is_valid(g, got, max(map(max, exp)))

[[1, 0], [2, 1]]
[[1, 1, 0], [0, 1, 1], [1, 2, 2]]


## 10. Vì sao thuật toán đúng

Gọi `d(c)` = khoảng cách (số bước 4 hướng) từ ô `c` tới ô nước gần nhất.

**Bổ đề 1 (cận trên).** Mọi phép gán hợp lệ có `H[c] ≤ d(c)`: dọc theo đường ngắn nhất `c = c_0, c_1, …, c_d = w` (ô nước), `H[c_k] ≤ H[c_{k+1}] + 1` và `H[w] = 0` ⇒ `H[c] ≤ d`.

**Bổ đề 2 (hợp lệ).** `H = d` hợp lệ: `d(nước) = 0`, `d ≥ 0`, và với hai ô kề `u, v`: `d(u) ≤ d(v) + 1` (đi từ `u` sang `v` rồi theo đường của `v`) và ngược lại ⇒ `|d(u) − d(v)| ≤ 1`.

⇒ `max H = max d` là tối ưu (không phép gán nào có ô cao hơn `d` của nó). ∎

**BFS tính đúng `d`:** BFS đa nguồn tương đương BFS từ một siêu nguồn ảo nối tới mọi ô nước; BFS trên đồ thị không trọng số cho khoảng cách ngắn nhất.

## 11. Phân tích Time / Space Complexity

Gọi `N = m·n ≤ 10⁶`, `W` = số ô nước.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn Manhattan | `O(N · W)` | `O(W)` |
| 2. BFS đa nguồn | `O(N)` | `O(N)` |
| 3. DP hai lượt | `O(N)` | `O(1)` phụ |
| 4. BFS theo tầng | `O(N)` | `O(N)` |

In [13]:
rng = random.Random(1765)
m = n = 400
big = [[1 if rng.random() < 0.001 else 0 for _ in range(n)] for _ in range(m)]
big[0][0] = 1
for name, f in [("BFS", Solution().highestPeak), ("DP", peak_dp), ("Layers", SolutionLayers().highestPeak)]:
    t = time.perf_counter(); H = f(big); dt = time.perf_counter() - t
    print(f"{name:<7} max={max(map(max, H))} ({dt*1000:.0f} ms cho 400x400)")

BFS     max=63 (80 ms cho 400x400)
DP      max=63 (39 ms cho 400x400)


Layers  max=63 (77 ms cho 400x400)


## 12. Bộ test

Mọi cách được kiểm bằng `is_valid` (đáp án nhiều lời giải) và đối chiếu giá trị với vét cạn (vì `H = d` là duy nhất cho cách gán "cao nhất từng ô", các cách của ta đều trả đúng ma trận khoảng cách).

In [14]:
ALL = {"bfs": peak_bfs, "dp": peak_dp, "layers": SolutionLayers().highestPeak, "submit": Solution().highestPeak}
def check(g):
    ref = peak_bruteforce(g); best = max(map(max, ref))
    for k, f in ALL.items():
        H = f(g)
        assert is_valid(g, H, best), (k, g)
        assert H == ref, (k, g)
    return best

for g, exp in examples:
    assert check(g) == max(map(max, exp))
edge = [[[1]], [[1, 0, 0, 0]], [[0], [0], [1]], [[1, 1], [1, 1]], [[0, 0, 0], [0, 1, 0], [0, 0, 0]]]
print("edge max:", [check(g) for g in edge])
rng = random.Random(17650)
for _ in range(600):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[1 if rng.random() < rng.choice([0.05, 0.2, 0.6]) else 0 for _ in range(n)] for _ in range(m)]
    g[rng.randrange(m)][rng.randrange(n)] = 1                # ít nhất một ô nước
    check(g)
print("Ví dụ, edge case, 600 lưới ngẫu nhiên: OK")

edge max: [0, 3, 2, 0, 2]
Ví dụ, edge case, 600 lưới ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — BFS riêng từ **một** ô nước (hoặc lần lượt từng ô và ghi đè)

In [15]:
def wrong_single_source(isWater):
    m, n = len(isWater), len(isWater[0])
    si, sj = next((i, j) for i in range(m) for j in range(n) if isWater[i][j])
    H = [[-1] * n for _ in range(m)]; H[si][sj] = 0; q = deque([(si, sj)])
    while q:
        i, j = q.popleft()
        for x, y in neighbors(i, j, m, n):
            if H[x][y] == -1:
                H[x][y] = 0 if isWater[x][y] else H[i][j] + 1; q.append((x, y))
    return H

g, exp = examples[1]
H = wrong_single_source(g)
print("sai :", H, "| hợp lệ?", is_valid(g, H, 2))

sai : [[2, 1, 0], [0, 2, 1], [4, 3, 2]] | hợp lệ? False


**Vì sao sai:** khoảng cách phải tính tới ô nước **gần nhất**, không phải một ô cố định; gán ô nước = 0 giữa chừng còn tạo bước nhảy chênh > 1. **Sửa:** đẩy **mọi** ô nước vào hàng đợi trước khi BFS.

### Lỗi 2 — Đánh dấu "đã thăm" khi **lấy ra** thay vì khi **đẩy vào**

In [16]:
def slow_mark_on_pop(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]; q = deque(); pushes = 0
    for i in range(m):
        for j in range(n):
            if isWater[i][j]: q.append((i, j, 0)); pushes += 1
    while q:
        i, j, d = q.popleft()
        if H[i][j] != -1: continue
        H[i][j] = d
        for x, y in neighbors(i, j, m, n):
            if H[x][y] == -1: q.append((x, y, d + 1)); pushes += 1
    return H, pushes

g = [[1] + [0] * 59 for _ in range(60)]
H, pushes = slow_mark_on_pop(g)
print("đúng kết quả?", H == Solution().highestPeak(g), "| số lần đẩy:", pushes, "so với số ô:", 60 * 60)

đúng kết quả? True | số lần đẩy: 7140 so với số ô: 3600


**Vì sao không tốt:** kết quả vẫn đúng nhưng một ô có thể bị đẩy nhiều lần (tới 4 lần) ⇒ tốn gấp nhiều lần bộ nhớ/thời gian, dễ TLE/MLE với `10⁶` ô. **Sửa:** gán `H[x][y]` ngay khi đẩy.

### Lỗi 3 — DP chỉ một lượt

In [17]:
def wrong_one_pass(isWater):
    m, n = len(isWater), len(isWater[0]); INF = m + n
    H = [[0 if isWater[i][j] else INF for j in range(n)] for i in range(m)]
    for i in range(m):
        for j in range(n):
            if i > 0: H[i][j] = min(H[i][j], H[i - 1][j] + 1)
            if j > 0: H[i][j] = min(H[i][j], H[i][j - 1] + 1)
    return H

g, exp = examples[0]
print("sai :", wrong_one_pass(g), "| đúng:", exp)

sai : [[4, 0], [4, 1]] | đúng: [[1, 0], [2, 1]]


**Vì sao sai:** một lượt chỉ lan truyền từ trên/trái; ô `(0,0)` ở bên **trái** ô nước `(0,1)` không nhận được giá trị (còn `INF = 4`). **Sửa:** thêm lượt ngược.

### Lỗi 4 — Dùng `list.pop(0)` làm hàng đợi

In [18]:
def slow_list_queue(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]; q = []
    for i in range(m):
        for j in range(n):
            if isWater[i][j]: H[i][j] = 0; q.append((i, j))
    while q:
        i, j = q.pop(0)                                    # O(len(q)) mỗi lần
        for x, y in neighbors(i, j, m, n):
            if H[x][y] == -1: H[x][y] = H[i][j] + 1; q.append((x, y))
    return H

g = [[1 if j < 199 else 0 for j in range(200)] for _ in range(200)]     # rất nhiều ô nước -> hàng đợi ban đầu ~40000 phần tử
for f in (slow_list_queue, Solution().highestPeak):
    t = time.perf_counter(); f(g); print(getattr(f, "__name__", "?"), f"{(time.perf_counter()-t)*1000:.0f} ms")

slow_list_queue 141 ms


highestPeak 19 ms


**Vì sao không tốt:** `pop(0)` dịch toàn bộ list ⇒ `O(N²)` tệ nhất — lộ rõ khi hàng đợi dài (nhiều ô nước). **Sửa:** `deque.popleft()` hoặc BFS theo tầng (Cách 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chênh lệch kề nhau ≤ 1, nguồn cố định = 0, làm giá trị lớn nhất" ⇒ giá trị tối ưu = **khoảng cách tới nguồn gần nhất**.
- Nhiều nguồn ⇒ **BFS đa nguồn** (đẩy hết nguồn vào hàng đợi trước).
- Lưới không vật cản + khoảng cách 4 hướng ⇒ DP hai lượt cũng được.
- Đánh dấu khi đẩy; dùng `deque`.
- Bài nhiều đáp án ⇒ viết **hàm kiểm tra hợp lệ**.

## 15. Biến thể của bài

1. **542 01 Matrix**: cùng bài (khoảng cách tới ô 0 gần nhất).
2. **994 Rotting Oranges**: BFS đa nguồn theo thời gian, có ô trống/vật cản.
3. **1162 As Far from Land as Possible**: max khoảng cách tới đất gần nhất.
4. **286 Walls and Gates** (Premium): BFS đa nguồn từ các cổng, có tường.
5. **Có vật cản** (ô không đi qua được): bỏ DP hai lượt, chỉ dùng BFS (Bài tập 5).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | min Manhattan tới mọi ô nước | `O(N·W)` | `O(W)` | Kiểm chứng |
| 2. BFS đa nguồn | Lan truyền từ mọi nguồn | `O(N)` | `O(N)` | **Bản nộp**, tổng quát (có vật cản) |
| 3. DP hai lượt | Lan truyền theo hai hướng quét | `O(N)` | `O(1)` | Lưới không vật cản |
| 4. BFS theo tầng | Tầng = chiều cao | `O(N)` | `O(N)` | Dễ trace |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[1,0,0]]`; (b) `[[0,0],[0,1]]`; (c) `[[1,0,1]]`; (d) `[[0,0,0],[0,0,0],[0,0,1]]` — cho max chiều cao.

**Bài 2 (tính tay):** Tính ma trận chiều cao cho `[[0,0,0,0],[0,1,0,0],[0,0,0,1]]`.

**Bài 3 (điền TODO):** Hoàn thiện `multi_source_init(isWater)` trả `(H, q)` sau bước khởi tạo BFS.

**Bài 4 (tự viết lại):** Viết `farthest_land(isWater)` trả **tọa độ** một ô đất cao nhất và chiều cao của nó (hoặc `None` nếu không có đất).

**Bài 5 (mini challenge):** Có thêm ô **núi đá** `2` (không đi qua được, chiều cao bỏ qua và ghi `−1`). Viết `peak_with_rocks(grid)`; ô đất không tới được nước thì ghi `−1`.

In [19]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def multi_source_init(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]
    q = deque()
    # TODO: với mọi ô nước: H = 0 và đẩy vào q
    return H, q

In [20]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def farthest_land(isWater):
    # TODO: chạy BFS, tìm ô có H lớn nhất mà isWater == 0
    pass

def peak_with_rocks(grid):
    # TODO: BFS đa nguồn từ các ô 1, không đi vào ô 2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: khoảng cách tới nước gần nhất. Mức 2: lấy max. Mức 3: (a) `2`; (b) `2`; (c) `1`; (d) `4`.

**Bài 2.** Mức 1: nước ở `(1,1)` và `(2,3)`. Mức 2: tính min Manhattan. Mức 3: `[[2,1,2,2],[1,0,1,1],[2,1,1,0]]`.

**Bài 3.** Mức 1: hai vòng `for`. Mức 2: `if isWater[i][j]`. Mức 3: `H[i][j] = 0; q.append((i, j))`.

**Bài 4.** Mức 1: `H = Solution().highestPeak(g)`. Mức 2: duyệt tìm max trên ô đất. Mức 3: nếu không có ô đất trả `None`.

**Bài 5.** Mức 1: khởi tạo như cũ, ô đá ghi `−1` và coi như đã thăm. Mức 2: BFS bỏ qua ô `2`. Mức 3: ô đất còn `−1` là không tới được. Đối chiếu bằng BFS từ từng ô nước.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
b1 = [max(map(max, Solution().highestPeak(g))) for g in [[[1, 0, 0]], [[0, 0], [0, 1]], [[1, 0, 1]], [[0, 0, 0], [0, 0, 0], [0, 0, 1]]]]
print("Bài 1:", b1); assert b1 == [2, 2, 1, 4]
b2 = Solution().highestPeak([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1]])
print("Bài 2:", b2); assert b2 == [[2, 1, 2, 2], [1, 0, 1, 1], [2, 1, 1, 0]]
def multi_source_init(isWater):
    m, n = len(isWater), len(isWater[0])
    H = [[-1] * n for _ in range(m)]; q = deque()
    for i in range(m):
        for j in range(n):
            if isWater[i][j]:
                H[i][j] = 0; q.append((i, j))
    return H, q
print("Bài 3:", multi_source_init(examples[1][0]))

Bài 1: [2, 2, 1, 4]
Bài 2: [[2, 1, 2, 2], [1, 0, 1, 1], [2, 1, 1, 0]]
Bài 3: ([[-1, -1, 0], [0, -1, -1], [-1, -1, -1]], deque([(0, 2), (1, 0)]))


In [22]:
# Bài 4, 5
def farthest_land(isWater):
    H = Solution().highestPeak(isWater)
    best = None
    for i, row in enumerate(H):
        for j, v in enumerate(row):
            if isWater[i][j] == 0 and (best is None or v > best[1]):
                best = ((i, j), v)
    return best

def peak_with_rocks(grid):
    m, n = len(grid), len(grid[0])
    H = [[-1] * n for _ in range(m)]; q = deque()
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1: H[i][j] = 0; q.append((i, j))
    while q:
        i, j = q.popleft()
        for x, y in neighbors(i, j, m, n):
            if grid[x][y] == 0 and H[x][y] == -1:
                H[x][y] = H[i][j] + 1; q.append((x, y))
    return H

def rocks_brute(grid):
    m, n = len(grid), len(grid[0]); INF = float("inf")
    best = [[INF] * n for _ in range(m)]
    for a in range(m):
        for b in range(n):
            if grid[a][b] != 1: continue
            D = {(a, b): 0}; q = deque([(a, b)])
            while q:
                i, j = q.popleft()
                for x, y in neighbors(i, j, m, n):
                    if grid[x][y] == 0 and (x, y) not in D:
                        D[(x, y)] = D[(i, j)] + 1; q.append((x, y))
            for (i, j), d in D.items(): best[i][j] = min(best[i][j], d)
    return [[-1 if v == INF else v for v in row] for row in best]

assert farthest_land(examples[1][0]) == ((2, 1), 2) and farthest_land([[1]]) is None
rng = random.Random(9)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    g = [[rng.choice([0, 0, 0, 1, 2]) for _ in range(n)] for _ in range(m)]
    assert peak_with_rocks(g) == rocks_brute(g), g
print("Bài 4, 5: OK |", farthest_land(examples[1][0]), peak_with_rocks([[1, 2, 0], [0, 2, 0], [0, 0, 0]]))

Bài 4, 5: OK | ((2, 1), 2) [[0, -1, 6], [1, -1, 5], [2, 3, 4]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template BFS đa nguồn:**

```python
H = [[-1]*n for _ in range(m)]; q = deque()
for i, j in nguon: H[i][j] = 0; q.append((i, j))
while q:
    i, j = q.popleft()
    for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
        x, y = i+dx, j+dy
        if 0 <= x < m and 0 <= y < n and H[x][y] == -1:
            H[x][y] = H[i][j] + 1; q.append((x, y))
```

**Checklist:**
- [ ] Đẩy **mọi** ô nước vào hàng đợi trước khi BFS?
- [ ] Đánh dấu (gán `H`) khi **đẩy vào**?
- [ ] Dùng `deque.popleft()` (không `list.pop(0)`)?
- [ ] Kiểm tra biên đầy đủ?
- [ ] Đã test lưới 1×1, toàn nước, một dòng/một cột, nước ở góc?